# USGS M2M API — Search & Download Satellite Imagery

This notebook provides a complete workflow for interacting with the **USGS Machine-to-Machine (M2M) API** to search for and download satellite imagery (Landsat, Sentinel, NAIP, etc.).

### Workflow Overview
1. **Authenticate** with your USGS EROS API token
2. **Define an Area of Interest (AOI)** from GeoJSON (Point, LineString, or Polygon)
3. **Search available datasets** by name wildcard within your AOI
4. **Search scenes** within a chosen dataset and get a summary (image count, date range, etc.)
5. **Filter by date range** and review an updated summary
6. **Apply additional metadata filters** (cloud cover, etc.)
7. **Assemble a download list** and submit download requests
8. **Monitor & download** files asynchronously

### Prerequisites
- A registered USGS EROS account with **M2M access** enabled: https://ers.cr.usgs.gov/profile/access
- An **Application Token** generated from your EROS profile (preferred over password auth)
- Python 3.8+

In [53]:
# ── Install Dependencies ──────────────────────────────────────────────────────
# Run once — these are the only external packages required.
!pip install requests aiohttp aiofiles tqdm shapely python-dotenv leafmap --quiet

In [72]:
import requests
import json
import time
import leafmap
import asyncio
import aiohttp
import aiofiles
import os
from pathlib import Path
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
from shapely.geometry import shape
from dotenv import load_dotenv

# ── Load .env file ────────────────────────────────────────────────────────────
load_dotenv()

# ── M2M API Configuration ────────────────────────────────────────────────────
M2M_BASE_URL = "https://m2m.cr.usgs.gov/api/api/json/stable/"

def m2m_request(endpoint: str, payload: dict | None = None, api_key: str | None = None) -> dict:
    """Send a POST request to the USGS M2M API and return the JSON response."""
    url = M2M_BASE_URL + endpoint
    headers = {}
    if api_key:
        headers["X-Auth-Token"] = api_key
    response = requests.post(url, json=payload or {}, headers=headers)
    response.raise_for_status()
    result = response.json()
    if result.get("errorCode"):
        raise RuntimeError(f"M2M API error [{result['errorCode']}]: {result['errorMessage']}")
    return result

print("✔ Imports and helpers loaded.")

✔ Imports and helpers loaded.


## Step 1 — Authenticate with the M2M API

The M2M API uses **token-based authentication**. You have two options:

| Method | Endpoint | Notes |
|---|---|---|
| `login-token` | Username + Application Token | **Recommended** — generate a token at https://ers.cr.usgs.gov/profile/access |
| `login` | Username + Password | Deprecated by USGS — may be removed |

After a successful login, the API returns an **API key** (session token) that must be passed as the `X-Auth-Token` header on all subsequent requests.

> **Security tip:** Never hard-code credentials. The cell below reads from environment variables or prompts you interactively.

In [73]:
# ── Authenticate ──────────────────────────────────────────────────────────────
# Credentials are loaded from the .env file via python-dotenv.
# Expected .env variables: USGS_USERNAME, USGSTOKEN

import getpass

usgs_username = os.environ.get("USGS_USERNAME") or input("USGS Username: ")
usgs_token    = os.environ.get("USGSTOKEN")    or getpass.getpass("USGS Application Token: ")

login_payload = {
    "username": usgs_username,
    "token": usgs_token,
}

login_result = m2m_request("login-token", login_payload)
API_KEY = login_result["data"]
print(f"✔ Authenticated as '{usgs_username}'  (API key: {API_KEY[:8]}…)")

✔ Authenticated as 'mapninja'  (API key: eyJjaWQi…)


## Step 2 — Define Your Area of Interest (AOI)

Provide your AOI as a **GeoJSON geometry** — the API accepts **Point**, **LineString**, or **Polygon** types.

The cell below includes a sample polygon over Denver, CO. Replace it with your own GeoJSON or load one from a file.

**Supported geometry types:**
- `Point` — a single coordinate `[longitude, latitude]`
- `LineString` — a list of coordinates forming a line
- `Polygon` — a list of rings (outer ring first), each a list of `[lon, lat]` pairs

> **Note:** Coordinates must be in **WGS 84 (EPSG:4326)** — longitude first, latitude second (GeoJSON standard).

In [74]:
# ── Define AOI ────────────────────────────────────────────────────────────────
# Option A: Paste inline GeoJSON geometry (Point, LineString, or Polygon).
# Option B: Load from a .geojson file  →  set GEOJSON_FILE below.

GEOJSON_FILE = "castle_2020.geojson"  # e.g. "my_aoi.geojson"

if GEOJSON_FILE and Path(GEOJSON_FILE).exists():
    with open(GEOJSON_FILE) as f:
        geojson_input = json.load(f)
    # Handle both FeatureCollection / Feature / bare Geometry
    if geojson_input.get("type") == "FeatureCollection":
        aoi_geometry = geojson_input["features"][0]["geometry"]
    elif geojson_input.get("type") == "Feature":
        aoi_geometry = geojson_input["geometry"]
    else:
        aoi_geometry = geojson_input
else:
    # ── Sample AOI: Polygon around Denver, CO ─────────────────────────────
    aoi_geometry = {
        "type": "Polygon",
        "coordinates": [[
            [-105.10, 39.85],
            [-104.90, 39.85],
            [-104.90, 39.65],
            [-105.10, 39.65],
            [-105.10, 39.85],
        ]]
    }

# Validate and summarise
aoi_shape = shape(aoi_geometry)
print(f"AOI type : {aoi_geometry['type']}")
print(f"Bounds   : {aoi_shape.bounds}")
print(f"Area     : {aoi_shape.area:.6f} sq degrees")

# ── Build the M2M spatial filter ──────────────────────────────────────────────
def build_spatial_filter(geom: dict) -> dict:
    """Convert a GeoJSON geometry into an M2M SpatialFilterGeoJson dict."""
    return {
        "filterType": "geojson",
        "geoJson": geom,
    }

spatial_filter = build_spatial_filter(aoi_geometry)
print(f"\n✔ Spatial filter ready ({aoi_geometry['type']})")

AOI type : MultiPolygon
Bounds   : (-118.84803118654672, 36.07840888914181, -118.29978622640292, 36.3891715450926)
Area     : 0.070328 sq degrees

✔ Spatial filter ready (MultiPolygon)


## Step 3 — Search Available Datasets (Wildcard)

Use the `dataset-search` endpoint to find datasets whose name matches a wildcard pattern within your AOI.

For example:
- `"landsat_*"` → all Landsat datasets
- `"*sentinel*"` → Sentinel collections
- `"naip"` → NAIP aerial imagery
- `"*"` → everything available for the AOI

The API uses `datasetName` as a wildcard filter (case-insensitive). We also pass the spatial filter so only datasets with data overlapping the AOI are returned.

In [75]:
# ── Dataset Search ────────────────────────────────────────────────────────────
# Change the wildcard pattern to explore different data collections.

DATASET_WILDCARD = "naip"   # ← Change this to search for other datasets

dataset_search_payload = {
    "datasetName": DATASET_WILDCARD,
    "spatialFilter": spatial_filter,
    "publicOnly": False,           # include datasets that require login
}

ds_result = m2m_request("dataset-search", dataset_search_payload, api_key=API_KEY)
datasets = ds_result.get("data") or []

print(f"Found {len(datasets)} dataset(s) matching '{DATASET_WILDCARD}':\n")
print(f"{'#':<4} {'Dataset Name':<45} {'Dataset Alias':<40} {'Abstract (first 80 chars)'}")
print("─" * 130)
for i, ds in enumerate(datasets, 1):
    abstract = (ds.get("abstractText") or "")[:80].replace("\n", " ")
    name  = ds.get("datasetName", "N/A")
    alias = ds.get("datasetAlias", "N/A")
    print(f"{i:<4} {name:<45} {alias:<40} {abstract}")

# Store for later use — key by datasetName, falling back to datasetAlias
def _ds_key(ds: dict) -> str | None:

    return ds.get("datasetName") or ds.get("datasetAlias")

available_datasets = {_ds_key(ds): ds for ds in datasets if _ds_key(ds)}


Found 1 dataset(s) matching 'naip':

#    Dataset Name                                  Dataset Alias                            Abstract (first 80 chars)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
1    N/A                                           naip                                     The National Agriculture Imagery Program (NAIP) acquires aerial imagery during t


## Step 4 — Scene Search & Summary for a Chosen Dataset

Select one dataset from the list above and search for all scenes that intersect your AOI.

The `scene-search` endpoint returns scene metadata including:
- **Entity ID** and **Display ID** (scene identifiers)
- **Acquisition date**
- **Cloud cover** (where applicable)
- **Spatial coverage** (browse URLs, footprint)

We'll paginate through all results and print a summary: total scene count, date range, cloud cover statistics, etc.

In [76]:
# ── Scene Search ──────────────────────────────────────────────────────────────
# Set the dataset you want to query (use a name from the list above).

DATASET_NAME = "naip"   # ← Change to any dataset name/alias from Step 3

# ── Helper: paginated scene search ───────────────────────────────────────────
def search_all_scenes(dataset_name: str, scene_filter: dict, api_key: str,
                      max_results_per_page: int = 100) -> list[dict]:
    """Paginate through scene-search and return all scene records."""
    all_scenes = []
    starting_number = 1
    total_hits = None

    while True:
        payload = {
            "datasetName": dataset_name,
            "sceneFilter": scene_filter,
            "maxResults": max_results_per_page,
            "startingNumber": starting_number,
            "metadataType": "full",
        }
        result = m2m_request("scene-search", payload, api_key=api_key)
        data = result["data"]
        total_hits = data.get("totalHits", total_hits)
        results = data.get("results", [])
        if not results:
            break
        all_scenes.extend(results)
        print(f"  Retrieved {len(all_scenes):,} / {total_hits:,} scenes …", end="\r")
        starting_number += max_results_per_page
        if len(all_scenes) >= total_hits:
            break

    print(f"  Retrieved {len(all_scenes):,} / {total_hits:,} scenes — done.")
    return all_scenes

# ── Initial (unfiltered) scene search ────────────────────────────────────────
scene_filter_base = {
    "spatialFilter": spatial_filter,
}

print(f"Searching scenes in '{DATASET_NAME}' …")
all_scenes = search_all_scenes(DATASET_NAME, scene_filter_base, API_KEY)

# ── Print summary ────────────────────────────────────────────────────────────
def print_scene_summary(scenes: list[dict], label: str = "Scene Summary"):
    """Print a concise summary of a list of scene records."""
    print(f"\n{'═' * 60}")
    print(f" {label}")
    print(f"{'═' * 60}")
    if not scenes:
        print("  No scenes found.")
        return

    dates = []
    cloud_covers = []
    for s in scenes:
        # Parse acquisition date
        acq = s.get("temporalCoverage", {}).get("startDate") or s.get("publishDate")
        if acq:
            try:
                dates.append(datetime.fromisoformat(acq.replace("Z", "+00:00")))
            except Exception:
                pass
        # Cloud cover from metadata
        cc = s.get("cloudCover")
        if cc is not None:
            try:
                cloud_covers.append(float(cc))
            except (ValueError, TypeError):
                pass

    print(f"  Total scenes       : {len(scenes):,}")
    if dates:
        print(f"  Date range         : {min(dates).date()} → {max(dates).date()}")
        years = Counter(d.year for d in dates)
        print(f"  Scenes per year    : {dict(sorted(years.items()))}")
    if cloud_covers:
        avg_cc = sum(cloud_covers) / len(cloud_covers)
        print(f"  Cloud cover range  : {min(cloud_covers):.1f}% – {max(cloud_covers):.1f}%")
        print(f"  Cloud cover mean   : {avg_cc:.1f}%")
        print(f"  Scenes < 20% cloud : {sum(1 for c in cloud_covers if c < 20):,}")
    print(f"{'═' * 60}\n")

print_scene_summary(all_scenes, f"All scenes in '{DATASET_NAME}'")

Searching scenes in 'naip' …
  Retrieved 383 / 383 scenes — done.

════════════════════════════════════════════════════════════
 All scenes in 'naip'
════════════════════════════════════════════════════════════
  Total scenes       : 383
  Date range         : 2005-07-01 → 2022-07-07
  Scenes per year    : {2005: 43, 2009: 43, 2010: 43, 2012: 43, 2014: 43, 2016: 42, 2018: 42, 2020: 42, 2022: 42}
════════════════════════════════════════════════════════════



## Step 5 — Filter by Date Range

Narrow results to a specific **acquisition date window**. The M2M API accepts an `acquisitionFilter` with `start` and `end` dates (format: `YYYY-MM-DD`).

This re-runs the scene search with the date constraint and prints an updated summary so you can compare before/after counts.

In [77]:
# ── Filter by Date Range ──────────────────────────────────────────────────────
# Adjust these dates to your project needs.

DATE_START = "2019-01-01"
DATE_END   = "2023-12-31"

acquisition_filter = {
    "start": DATE_START,
    "end":   DATE_END,
}

scene_filter_dates = {
    "spatialFilter": spatial_filter,
    "acquisitionFilter": acquisition_filter,
}

print(f"Searching '{DATASET_NAME}' for {DATE_START} → {DATE_END} …")
scenes_date_filtered = search_all_scenes(DATASET_NAME, scene_filter_dates, API_KEY)
print_scene_summary(scenes_date_filtered, f"Date-filtered scenes ({DATE_START} → {DATE_END})")

Searching 'naip' for 2019-01-01 → 2023-12-31 …
  Retrieved 84 / 84 scenes — done.

════════════════════════════════════════════════════════════
 Date-filtered scenes (2019-01-01 → 2023-12-31)
════════════════════════════════════════════════════════════
  Total scenes       : 84
  Date range         : 2020-07-26 → 2022-07-07
  Scenes per year    : {2020: 42, 2022: 42}
════════════════════════════════════════════════════════════



## Step 5b — Inspect Metadata for the Most Recent Scene

Each scene record returned by the API contains rich metadata, including:

| Field | Description |
|---|---|
| `entityId` / `displayId` | Unique scene identifiers |
| `temporalCoverage` | Acquisition start/end dates |
| `spatialCoverage` | Bounding coordinates / footprint |
| `cloudCover` | Percentage of cloud cover (if applicable) |
| `publishDate` | Date the scene was published to the archive |
| `browse` | List of browse/thumbnail image URLs |
| `metadata` | Full list of metadata key–value pairs (sensor, path/row, etc.) |

The cell below finds the **most recent scene** from the date-filtered results and prints all available metadata fields.

In [78]:
# ── Most Recent Scene Metadata ────────────────────────────────────────────────
# Use the date-filtered results; fall back to all_scenes if empty.
scenes_to_inspect = scenes_date_filtered or all_scenes

if not scenes_to_inspect:
    print("No scenes available to inspect.")
else:
    # Sort by acquisition date (most recent first)
    def _acq_date(s):
        raw = (s.get("temporalCoverage") or {}).get("startDate") or s.get("publishDate") or ""
        try:
            return datetime.fromisoformat(raw.replace("Z", "+00:00"))
        except Exception:
            return datetime.min

    most_recent = max(scenes_to_inspect, key=_acq_date)

    print(f"{'═' * 70}")
    print(f" Most Recent Scene")
    print(f"{'═' * 70}")

    # ── Top-level fields ──────────────────────────────────────────────────
    highlight_keys = [
        "entityId", "displayId", "datasetName", "publishDate",
        "cloudCover", "temporalCoverage", "spatialCoverage",
    ]
    for key in highlight_keys:
        val = most_recent.get(key)
        if val is not None:
            print(f"  {key:<25}: {val}")

    # ── Browse / thumbnail URLs ───────────────────────────────────────────
    browse = most_recent.get("browse") or []
    if browse:
        print(f"\n  Browse images ({len(browse)}):")
        for b in browse:
            print(f"    • {b.get('browseName', 'N/A')}: {b.get('browsePath', 'N/A')}")

    # ── Full metadata list ────────────────────────────────────────────────
    metadata = most_recent.get("metadata") or []
    if metadata:
        print(f"\n  Metadata fields ({len(metadata)}):")
        print(f"  {'Field':<40} {'Value'}")
        print(f"  {'─' * 40} {'─' * 50}")
        for entry in metadata:
            field_name = entry.get("fieldName", "?")
            value = str(entry.get("value", ""))[:80]
            print(f"  {field_name:<40} {value}")

    # ── Dump remaining top-level keys not already shown ───────────────────
    shown = set(highlight_keys) | {"browse", "metadata"}
    extra = {k: v for k, v in most_recent.items() if k not in shown and v is not None}
    if extra:
        print(f"\n  Other fields:")
        for k, v in extra.items():
            print(f"  {k:<25}: {json.dumps(v, default=str)[:120]}")

    print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
 Most Recent Scene
══════════════════════════════════════════════════════════════════════
  entityId                 : 3234534
  displayId                : M_3611835_SE_11_060_20220707
  publishDate              : 2023-04-10 13:52:37.622668-05
  temporalCoverage         : {'endDate': '2022-07-07 00:00:00-05', 'startDate': '2022-07-07 00:00:00-05'}
  spatialCoverage          : {'type': 'Polygon', 'coordinates': [[[-118.6896749, 36.372311], [-118.621411, 36.3732554], [-118.6228027, 36.4401944], [-118.6911249, 36.4392471], [-118.6896749, 36.372311]]]}

  Browse images (1):
    • Standard Browse: https://ims.cr.usgs.gov/browse/naip/fullres/CA/2022/202204_califorina_0x6000m_utm_cnir/36118/m_3611835_se_11_060_20220707.jpg

  Metadata fields (33):
  Field                                    Value
  ──────────────────────────────────────── ──────────────────────────────────────────────────
  State                            

## Step 5c — Spatial Bounds of the Most Recent Scene

The `spatialBounds` (or `spatialCoverage`) field on each scene record describes its geographic footprint. The cell below extracts and prints the bounding coordinates for the most recent scene identified above.

In [79]:
# ── Spatial Bounds of the Most Recent Scene ───────────────────────────────────

def get_spatial_bounds(scene: dict) -> dict | None:
    """Extract spatial bounds from a scene record.
    The API may store them under 'spatialBounds' or 'spatialCoverage'."""
    return scene.get("spatialBounds") or scene.get("spatialCoverage")

bounds = get_spatial_bounds(most_recent)

print(f"{'═' * 60}")
print(f" Spatial Bounds — {most_recent.get('displayId', most_recent.get('entityId', 'N/A'))}")
print(f"{'═' * 60}")

if bounds:
    print(f"  Raw value: {json.dumps(bounds, indent=2)}")
else:
    # Fall back: look inside metadata for corner coordinates
    print("  No top-level spatialBounds/spatialCoverage found.")
    print("  Checking metadata for coordinate fields …")
    coord_fields = {}
    for entry in (most_recent.get("metadata") or []):
        fn = (entry.get("fieldName") or "").lower()
        if any(k in fn for k in ["corner", "latitude", "longitude", "bound", "north", "south", "east", "west"]):
            coord_fields[entry["fieldName"]] = entry.get("value")
    if coord_fields:
        for k, v in coord_fields.items():
            print(f"  {k:<40}: {v}")
    else:
        print("  No coordinate metadata found.")

print(f"{'═' * 60}")

════════════════════════════════════════════════════════════
 Spatial Bounds — M_3611835_SE_11_060_20220707
════════════════════════════════════════════════════════════
  Raw value: {
  "type": "Polygon",
  "coordinates": [
    [
      [
        -118.6911249,
        36.372311
      ],
      [
        -118.6911249,
        36.4401944
      ],
      [
        -118.621411,
        36.4401944
      ],
      [
        -118.621411,
        36.372311
      ],
      [
        -118.6911249,
        36.372311
      ]
    ]
  ]
}
════════════════════════════════════════════════════════════


## Step 5d — Visualize All Filtered Scene Footprints on a Map

Use **leafmap** to plot the spatial bounds of every scene from the date-filtered results on an interactive map. Each footprint is drawn as a rectangle with a popup showing the scene's Display ID and acquisition date.

> `leafmap` is installed below if not already present. It wraps **folium** / **ipyleaflet** and makes it easy to add GeoJSON layers.

In [ ]:
import folium
from IPython.display import display

# ── Build GeoJSON features from scene spatial bounds ─────────────────────────
def scene_bounds_to_geojson(scenes: list[dict]) -> dict:
    """Convert a list of scene records into a GeoJSON FeatureCollection of
    bounding-box polygons.  Handles both the 'spatialBounds' and
    'spatialCoverage' keys the M2M API may return."""
    features = []
    for s in scenes:
        sb = s.get("spatialBounds") or s.get("spatialCoverage")
        if not sb:
            continue

        # The API may return a GeoJSON geometry directly …
        if "type" in sb and "coordinates" in sb:
            geom = sb
        else:
            # … or a dict with corner coordinates
            try:
                coords = sb.get("coordinates") or sb
                if isinstance(coords, list):
                    geom = {"type": "Polygon", "coordinates": coords}
                else:
                    n = float(coords.get("north") or coords.get("upperLeftY") or coords.get("maxY", 0))
                    s_lat = float(coords.get("south") or coords.get("lowerRightY") or coords.get("minY", 0))
                    e = float(coords.get("east") or coords.get("lowerRightX") or coords.get("maxX", 0))
                    w = float(coords.get("west") or coords.get("upperLeftX") or coords.get("minX", 0))
                    geom = {
                        "type": "Polygon",
                        "coordinates": [[[w, s_lat], [e, s_lat], [e, n], [w, n], [w, s_lat]]],
                    }
            except Exception:
                continue

        acq = (s.get("temporalCoverage") or {}).get("startDate") or s.get("publishDate") or "N/A"
        display_id = s.get("displayId") or s.get("entityId") or "N/A"

        features.append({
            "type": "Feature",
            "geometry": geom,
            "properties": {
                "displayId": display_id,
                "acquisitionDate": acq,
                "cloudCover": s.get("cloudCover"),
            },
        })

    return {"type": "FeatureCollection", "features": features}


# ── Create the map ───────────────────────────────────────────────────────────
scenes_geojson = scene_bounds_to_geojson(scenes_date_filtered or all_scenes)
print(f"Mapped {len(scenes_geojson['features'])} scene footprint(s).")

# Compute map center and bounds from footprints + AOI
from shapely.geometry import shape as shp
from shapely.ops import unary_union

all_geoms = [shp(aoi_geometry)]
for f in scenes_geojson["features"]:
    try:
        all_geoms.append(shp(f["geometry"]))
    except Exception:
        pass

combined = unary_union(all_geoms)
minx, miny, maxx, maxy = combined.bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# Add AOI outline
folium.GeoJson(
    {"type": "FeatureCollection",
     "features": [{"type": "Feature", "geometry": aoi_geometry, "properties": {"name": "AOI"}}]},
    name="AOI",
    style_function=lambda x: {"color": "red", "weight": 2, "fillOpacity": 0.05},
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Layer"]),
).add_to(m)

# Add scene footprints with popups
if scenes_geojson["features"]:
    folium.GeoJson(
        scenes_geojson,
        name="Scene Footprints",
        style_function=lambda x: {"color": "#3388ff", "weight": 1.5, "fillOpacity": 0.15},
        tooltip=folium.GeoJsonTooltip(
            fields=["displayId", "acquisitionDate", "cloudCover"],
            aliases=["Display ID", "Acquired", "Cloud %"],
        ),
    ).add_to(m)

# Fit to bounds
m.fit_bounds([[miny, minx], [maxy, maxx]])
folium.LayerControl().add_to(m)

# Save to HTML and display via IFrame (works in VS Code without "Trust Notebook")
from IPython.display import IFrame

map_path = "scene_footprints_map.html"
m.save(map_path)
print(f"Map saved to {map_path}")
IFrame(map_path, width="100%", height=600)

## Step 6 — Query Download Options

Before downloading, you must ask the API **which file products are available** for each scene. The `download-options` endpoint returns a list of downloadable products for the given scene Entity IDs.

### `download-options` — Request

| Parameter | Type | Description |
|---|---|---|
| `datasetName` | `string` | The dataset name or alias (e.g. `"naip"`) |
| `entityIds` | `string[]` | List of scene Entity IDs to check |
| `listId` | `string` | *(Optional)* A pre-built scene list ID (alternative to `entityIds`) |

### `download-options` — Response (per product)

| Field | Description |
|---|---|
| `id` | Unique product ID (used later in `download-request`) |
| `entityId` | The scene this product belongs to |
| `productName` | Human-readable product name (e.g. *"Full Resolution"*, *"LandsatLook Quality Image"*) |
| `productCode` | Short product code |
| `available` | `true` if the file is ready for immediate download; `false` if it must be ordered first |
| `filesize` | Approximate size in bytes |
| `downloadSystem` | Delivery system (`"dds"` = direct download, `"tram"` = tape archive/order) |

> **Tip:** Only products with `available: true` and `downloadSystem: "dds"` can be downloaded immediately. Products on `"tram"` require an order step first — see Step 8.

In [81]:
# ── Query Download Options ─────────────────────────────────────────────────────
# Collect Entity IDs from the date-filtered scenes (or all scenes if no filter).
target_scenes = scenes_date_filtered or all_scenes

entity_ids = [s["entityId"] for s in target_scenes]
print(f"Querying download options for {len(entity_ids)} scene(s) in '{DATASET_NAME}' …\n")

download_options_payload = {
    "datasetName": DATASET_NAME,
    "entityIds": entity_ids,
}

dl_options_result = m2m_request("download-options", download_options_payload, api_key=API_KEY)
dl_options = dl_options_result.get("data") or []

# ── Summarize available products ──────────────────────────────────────────────
print(f"{'═' * 90}")
print(f" Download Options — {len(dl_options)} product(s) across {len(entity_ids)} scene(s)")
print(f"{'═' * 90}")
print(f"  {'Entity ID':<30} {'Product Name':<35} {'Available':<10} {'System':<8} {'Size (MB)':<12}")
print(f"  {'─' * 30} {'─' * 35} {'─' * 10} {'─' * 8} {'─' * 12}")

# Track products eligible for immediate download
downloadable_products = []

for opt in dl_options:
    eid = opt.get("entityId", "N/A")
    pname = (opt.get("productName") or "N/A")[:34]
    avail = opt.get("available", False)
    system = opt.get("downloadSystem", "N/A")
    size_bytes = opt.get("filesize", 0) or 0
    size_mb = size_bytes / (1024 * 1024) if size_bytes else 0.0

    flag = "✔" if avail else "✘"
    print(f"  {eid:<30} {pname:<35} {flag:<10} {system:<8} {size_mb:>9.1f}")

    if avail and system == "dds":
        downloadable_products.append(opt)

print(f"{'═' * 90}")
print(f"\n✔ {len(downloadable_products)} product(s) available for immediate download (DDS).")
if len(dl_options) - len(downloadable_products) > 0:
    tram_count = sum(1 for o in dl_options if o.get("downloadSystem") == "tram")
    print(f"  {tram_count} product(s) require ordering via TRAM (not immediately available).")

Querying download options for 84 scene(s) in 'naip' …

══════════════════════════════════════════════════════════════════════════════════════════
 Download Options — 168 product(s) across 84 scene(s)
══════════════════════════════════════════════════════════════════════════════════════════
  Entity ID                      Product Name                        Available  System   Size (MB)   
  ────────────────────────────── ─────────────────────────────────── ────────── ──────── ────────────
  2995814                        Full Resolution                     ✔          dds          412.5
  2995814                        Compressed                          ✔          dds           70.2
  3234534                        Full Resolution                     ✔          dds          439.6
  3234534                        Compressed                          ✘          dds            0.0
  2995815                        Full Resolution                     ✔          dds          415.6
  2995815 

## Step 7 — Request & Download Immediately Available (DDS) Products

This step handles products with `downloadSystem: "dds"` and `available: true` from Step 6 — these files live on disk and can be downloaded **right now**.

The `download-request` endpoint converts product selections into actual **download URLs**.

### `download-request` — Request

| Parameter | Type | Description |
|---|---|---|
| `downloads` | `object[]` | List of `{ "entityId": "…", "productId": "…" }` pairs |
| `label` | `string` | *(Optional)* A label for this download batch — useful for tracking in your EROS account |

### `download-request` — Response

| Field | Description |
|---|---|
| `availableDownloads` | Products ready **now** — each entry has a `url` field for direct HTTP download |
| `preparingDownloads` | Products being staged from tape (should be empty for DDS products) |
| `duplicateProducts` | Products already requested in a recent session (de-duplicated) |
| `numInvalidScenes` | Count of Entity IDs not found or ineligible |


> **Note:** Download URLs are **temporary** and expire quickly. The code cell below requests URLs and downloads them immediately.
>
> TRAM (tape-archived) products are handled separately in **Step 9**.

### Product type filter

Set `PRODUCT_TYPE` at the top of the code cell to control which products are downloaded:

| Value | Downloads |
|---|---|
| `"full"` | Only **Full Resolution** products |
| `"compressed"` | Only **Compressed** / reduced-resolution products |
| `"both"` | **All** available product types (default) |

In [83]:
# ── Request & Download DDS Products (Immediately Available) ───────────────────

DOWNLOAD_DIR = Path(f"./downloads/{DATASET_NAME}")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

MAX_CONCURRENT_DOWNLOADS = 5  # ← number of parallel file downloads

# ── Product type selection ────────────────────────────────────────────────────
# Choose which product types to download:
#   "full"       → Full Resolution only
#   "compressed" → Compressed / reduced-resolution only
#   "both"       → Download all available product types
PRODUCT_TYPE = "full"  # ← Change to "full", "compressed", or "both"

def _match_product_type(product_name: str, selection: str) -> bool:
    """Return True if the product name matches the user's selection."""
    name_lower = (product_name or "").lower()
    if selection == "both":
        return True
    elif selection == "full":
        return "full" in name_lower and "compress" not in name_lower
    elif selection == "compressed":
        return "compress" in name_lower or "reduced" in name_lower
    return True  # fallback: include everything

# ── Separate DDS vs TRAM products from Step 6 ────────────────────────────────
dds_all = [opt for opt in downloadable_products]  # already filtered to DDS+available
tram_products = [opt for opt in dl_options
                 if opt.get("downloadSystem") == "tram" or not opt.get("available", False)]

# Apply product-type filter
dds_products = [opt for opt in dds_all
                if _match_product_type(opt.get("productName", ""), PRODUCT_TYPE)]

print(f"Product filter   : {PRODUCT_TYPE!r}")
print(f"DDS  (matched)   : {len(dds_products)} of {len(dds_all)} product(s)")
print(f"TRAM (archived)  : {len(tram_products)} product(s)  → handled in Step 9\n")

if dds_products:
    print("Selected DDS products:")
    for p in dds_products:
        print(f"  • {p.get('entityId','?')}  —  {p.get('productName','?')}")
    print()

# ── Async downloader (used by both Step 7 and Step 9) ────────────────────────
async def download_file(session: aiohttp.ClientSession, url: str, dest: Path,
                        pbar: tqdm) -> Path:
    """Download a single file with streaming and progress updates."""
    async with session.get(url) as resp:
        resp.raise_for_status()
        cd = resp.headers.get("Content-Disposition", "")
        if "filename=" in cd:
            fname = cd.split("filename=")[-1].strip('" ')
        else:
            fname = url.split("/")[-1].split("?")[0] or "download"
        filepath = dest / fname
        async with aiofiles.open(filepath, "wb") as f:
            async for chunk in resp.content.iter_chunked(1024 * 64):
                await f.write(chunk)
                pbar.update(len(chunk))
    return filepath


async def download_all(urls: list[str], dest: Path,
                       max_concurrent: int = MAX_CONCURRENT_DOWNLOADS):
    """Download a list of URLs concurrently."""
    sem = asyncio.Semaphore(max_concurrent)
    pbar = tqdm(total=0, unit="B", unit_scale=True,
                desc=f"Downloading ({max_concurrent} parallel)")

    async def _bounded(url):
        async with sem:
            return await download_file(session, url, dest, pbar)

    async with aiohttp.ClientSession() as session:
        tasks = [asyncio.ensure_future(_bounded(u)) for u in urls]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    pbar.close()
    return results


# ── Request DDS download URLs ─────────────────────────────────────────────────
if not dds_products:
    print("No DDS products to download. Skip to Step 9 for TRAM orders.")
    available_downloads = []
else:
    DDS_LABEL = f"dds_{DATASET_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    downloads_list = [
        {"entityId": opt["entityId"], "productId": opt["id"]}
        for opt in dds_products
    ]

    print(f"Requesting {len(downloads_list)} DDS download(s) (label: '{DDS_LABEL}') …\n")

    dl_request_result = m2m_request("download-request",
                                    {"downloads": downloads_list, "label": DDS_LABEL},
                                    api_key=API_KEY)
    dl_data = dl_request_result.get("data", {})

    available_downloads = dl_data.get("availableDownloads") or []
    duplicate_products  = dl_data.get("duplicateProducts") or []

    print(f"{'═' * 80}")
    print(f" DDS Download Request Results")
    print(f"{'═' * 80}")
    print(f"  Ready now  : {len(available_downloads)}")
    print(f"  Duplicates : {len(duplicate_products)}")
    print(f"{'═' * 80}")

    if available_downloads:
        print(f"\n  Download URLs:")
        for dl in available_downloads:
            eid = dl.get("entityId", "N/A")
            url = dl.get("url", "N/A")
            print(f"    • {eid}: {url[:100]}{'…' if len(url) > 100 else ''}")

    # ── Download DDS files now ────────────────────────────────────────────
    dds_urls = [dl["url"] for dl in available_downloads if dl.get("url")]
    if dds_urls:
        print(f"\nDownloading {len(dds_urls)} file(s) to {DOWNLOAD_DIR.resolve()} "
              f"({MAX_CONCURRENT_DOWNLOADS} parallel) …")
        results = await download_all(dds_urls, DOWNLOAD_DIR, MAX_CONCURRENT_DOWNLOADS)
        successes = [r for r in results if isinstance(r, Path)]
        failures  = [r for r in results if isinstance(r, Exception)]
        print(f"\n✔ {len(successes)} DDS file(s) downloaded successfully.")
        if failures:
            print(f"✘ {len(failures)} download(s) failed:")
            for err in failures:
                print(f"    {err}")
    else:
        print("\nNo DDS download URLs returned (files may already have been downloaded).")

Product filter   : 'full'
DDS  (matched)   : 84 of 126 product(s)
TRAM (archived)  : 42 product(s)  → handled in Step 9

Selected DDS products:
  • 2995814  —  Full Resolution
  • 3234534  —  Full Resolution
  • 2995815  —  Full Resolution
  • 3234535  —  Full Resolution
  • 2995822  —  Full Resolution
  • 3234542  —  Full Resolution
  • 2995823  —  Full Resolution
  • 3234543  —  Full Resolution
  • 2995840  —  Full Resolution
  • 3234560  —  Full Resolution
  • 2995841  —  Full Resolution
  • 3234561  —  Full Resolution
  • 2995842  —  Full Resolution
  • 3234562  —  Full Resolution
  • 2995843  —  Full Resolution
  • 3234563  —  Full Resolution
  • 2995844  —  Full Resolution
  • 3234564  —  Full Resolution
  • 2995845  —  Full Resolution
  • 3234565  —  Full Resolution
  • 2995846  —  Full Resolution
  • 3234566  —  Full Resolution
  • 2995847  —  Full Resolution
  • 3234567  —  Full Resolution
  • 2995850  —  Full Resolution
  • 3234570  —  Full Resolution
  • 2995851  —  Full Res


✔ 60 DDS file(s) downloaded successfully.
✘ 4 download(s) failed:
    
    
    
    


## Step 8 — Verify DDS Downloads

Quick sanity check: list the files downloaded in Step 7.

In [84]:
# ── Verify DDS Downloads ──────────────────────────────────────────────────────
if DOWNLOAD_DIR.exists():
    files = sorted(DOWNLOAD_DIR.iterdir())
    total_size = sum(f.stat().st_size for f in files if f.is_file())
    print(f"{'═' * 70}")
    print(f" Downloaded files in {DOWNLOAD_DIR}")
    print(f"{'═' * 70}")
    for fp in files:
        if fp.is_file():
            mb = fp.stat().st_size / (1024 * 1024)
            print(f"  {fp.name:<50} {mb:>8.1f} MB")
    print(f"{'─' * 70}")
    print(f"  {'Total':<50} {total_size / (1024 * 1024):>8.1f} MB")
    print(f"{'═' * 70}")
else:
    print(f"Download directory {DOWNLOAD_DIR} does not exist yet.")

print(f"\nTRAM products to handle in Step 9: {len(tram_products)}")

══════════════════════════════════════════════════════════════════════
 Downloaded files in downloads/naip
══════════════════════════════════════════════════════════════════════
  .DS_Store                                               0.0 MB
  m_3611835_se_11_060_20200802.ZIP                      412.5 MB
  m_3611835_se_11_060_20220707.ZIP                      439.6 MB
  m_3611835_sw_11_060_20200802.ZIP                      415.6 MB
  m_3611835_sw_11_060_20220703.ZIP                      442.2 MB
  m_3611837_se_11_060_20200802.ZIP                      420.2 MB
  m_3611837_se_11_060_20220704.ZIP                      429.6 MB
  m_3611837_sw_11_060_20200802.ZIP                      432.5 MB
  m_3611837_sw_11_060_20220704.ZIP                      416.0 MB
  m_3611842_ne_11_060_20200802.ZIP                      420.1 MB
  m_3611842_ne_11_060_20220704.ZIP                      442.3 MB
  m_3611842_nw_11_060_20200802.ZIP                      422.7 MB
  m_3611842_nw_11_060_20220704.ZIP        

## Step 9 — Request & Download TRAM (Tape-Archived) Products

**TRAM** (Tape Retrieval and Archive Management) is the USGS offline storage system. Older or less-frequently accessed data lives on tape and must be **staged to disk** before it can be downloaded. This process can take **minutes to hours** depending on queue depth.

### Workflow

1. Submit TRAM products via `download-request` — they appear in `preparingDownloads`
2. Poll `download-retrieve` with the same label — as files are staged, they move to `availableDownloads`
3. Download the URLs as they become available

### `download-retrieve` — Request / Response

| Parameter | Type | Description |
|---|---|---|
| `label` | `string` | The label used in `download-request` |

Returns the same `availableDownloads` / `preparingDownloads` structure. Poll until `preparingDownloads` is empty.

> **Tip:** If you have no TRAM products from Step 6, this cell will simply print a message and skip.
>
> For very large orders you can also use `order-submit` to place a formal order that USGS processes asynchronously and optionally emails you when complete.

In [66]:
# ── Request & Download TRAM Products ──────────────────────────────────────────

POLL_INTERVAL = 30   # seconds between download-retrieve polls
MAX_POLLS     = 60   # give up after ~30 min

if not tram_products:
    print("No TRAM products found — nothing to stage. ✔")
else:
    print(f"Submitting {len(tram_products)} TRAM product(s) for staging …\n")

    TRAM_LABEL = f"tram_{DATASET_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    tram_downloads_list = [
        {"entityId": opt["entityId"], "productId": opt["id"]}
        for opt in tram_products
    ]

    tram_request_result = m2m_request(
        "download-request",
        {"downloads": tram_downloads_list, "label": TRAM_LABEL},
        api_key=API_KEY,
    )
    tram_data = tram_request_result.get("data", {})

    tram_available  = tram_data.get("availableDownloads") or []
    tram_preparing  = tram_data.get("preparingDownloads") or []
    tram_duplicates = tram_data.get("duplicateProducts") or []

    print(f"{'═' * 80}")
    print(f" TRAM Download Request Results")
    print(f"{'═' * 80}")
    print(f"  Ready now  : {len(tram_available)}")
    print(f"  Staging    : {len(tram_preparing)}")
    print(f"  Duplicates : {len(tram_duplicates)}")
    print(f"{'═' * 80}")

    # Collect any URLs that are already available
    tram_urls = [dl["url"] for dl in tram_available if dl.get("url")]

    # ── Poll for staged files ────────────────────────────────────────────────
    if tram_preparing:
        print(f"\n⏳ {len(tram_preparing)} file(s) are staging from tape — "
              f"polling every {POLL_INTERVAL}s (max {MAX_POLLS} attempts) …\n")

        for attempt in range(1, MAX_POLLS + 1):
            time.sleep(POLL_INTERVAL)
            retrieve_result = m2m_request("download-retrieve",
                                          {"label": TRAM_LABEL}, api_key=API_KEY)
            rdata = retrieve_result.get("data", {})
            newly_ready     = rdata.get("availableDownloads") or []
            still_preparing = rdata.get("preparingDownloads") or []

            new_urls = [dl["url"] for dl in newly_ready if dl.get("url")]
            tram_urls.extend(new_urls)

            elapsed = attempt * POLL_INTERVAL
            print(f"  Poll {attempt}/{MAX_POLLS} ({elapsed}s): "
                  f"+{len(new_urls)} ready, {len(still_preparing)} still staging")

            if not still_preparing:
                print("  ✔ All TRAM files are now staged and ready.")
                break
        else:
            print(f"\n  ⚠ Stopped polling after {MAX_POLLS * POLL_INTERVAL}s. "
                  f"Re-run this cell later to check again, or use the USGS "
                  f"EarthExplorer web UI to monitor order '{TRAM_LABEL}'.")

    # ── Download staged TRAM files ───────────────────────────────────────────
    # Reuse the async downloader defined in Step 7
    if tram_urls:
        print(f"\nDownloading {len(tram_urls)} TRAM file(s) to {DOWNLOAD_DIR.resolve()} "
              f"({MAX_CONCURRENT_DOWNLOADS} parallel) …")
        tram_results = await download_all(tram_urls, DOWNLOAD_DIR, MAX_CONCURRENT_DOWNLOADS)
        successes = [r for r in tram_results if isinstance(r, Path)]
        failures  = [r for r in tram_results if isinstance(r, Exception)]
        print(f"\n✔ {len(successes)} TRAM file(s) downloaded successfully.")
        if failures:
            print(f"✘ {len(failures)} download(s) failed:")
            for err in failures:
                print(f"    {err}")
    elif tram_preparing:
        print("\nNo TRAM URLs available yet. Re-run this cell after staging completes.")
    else:
        print("\nAll TRAM products were duplicates or already downloaded.")

Submitting 1 TRAM product(s) for staging …

════════════════════════════════════════════════════════════════════════════════
 TRAM Download Request Results
════════════════════════════════════════════════════════════════════════════════
  Ready now  : 0
  Staging    : 0
  Duplicates : 0
════════════════════════════════════════════════════════════════════════════════

All TRAM products were duplicates or already downloaded.
